# 04 — Results: tables and figures for the paper

Turns the saved training output into the artifacts the paper reports. This
notebook **only aggregates what is on disk** — it never trains, and it never
invents a number that the parquets do not contain.

Training is still running, so every section reads what exists and says plainly
what is missing. A table built from a partial grid is labelled as partial.

**Inputs**
- `results/*.parquet` — one row per (disease, definition, h, test_year, model)
  with AUROC, AUPRC, and the test-fold size
- `predictions/*.parquet` — one row per municipality-month per model, with the
  true label and the predicted probability
- `data/master/labels.parquet` — the 16 outbreak labels, used to recover the
  municipality's outbreak state in the *current* month

**Outputs** — CSV tables under `<train dir>/tables/`, figures under `figuras/`.

## Section 0 — Setup

**Which training directory.** `data/train/` holds the first run, whose
`target_year` was computed as `ano + (mes == 12)` — correct at h = 1 but wrong
for every longer horizon, which put months that predict into the test year in the
training set. `data/train_fast/` is the corrected re-run. Everything below reads
the corrected one; point `TRAIN_DIR` at the old folder only to reproduce the
earlier numbers deliberately.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
MASTER_DIR = PROJECT_ROOT / "data" / "master"

# The corrected run. See the markdown above before changing this.
TRAIN_DIR = PROJECT_ROOT / "data" / "train_fast"

RESULTS_DIR = TRAIN_DIR / "results"
PRED_DIR = TRAIN_DIR / "predictions"
TABLES_DIR = TRAIN_DIR / "tables"
FIG_DIR = PROJECT_ROOT / "figuras"
for d in (TABLES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEST_YEARS = [2022, 2023, 2024]
COMBO = re.compile(r"^(\w+?)_(\w+)_h(\d+)\.parquet$")

# --- What finished training, and what has not ---
def available_combos(folder):
    found = {}
    for path in sorted(folder.glob("*.parquet")):
        m = COMBO.match(path.name)
        if m:
            found[(m.group(1), m.group(2), int(m.group(3)))] = path
    return found

results_on_disk = available_combos(RESULTS_DIR)
preds_on_disk = available_combos(PRED_DIR)
print(f"Reading from: {TRAIN_DIR.relative_to(PROJECT_ROOT)}")
print(f"Result files:      {len(results_on_disk)} of 96 combos")
print(f"Prediction files:  {len(preds_on_disk)}")

by_disease = {}
for (disease, definition, h) in results_on_disk:
    by_disease.setdefault(disease, set()).add((definition, h))
print("\nCombos finished, by disease (24 = complete: 4 definitions x 6 horizons):")
for disease in ("dengue", "zika", "chikungunya", "fa"):
    n = len(by_disease.get(disease, ()))
    state = "COMPLETE" if n == 24 else f"partial ({n}/24)" if n else "not started"
    print(f"  {disease:<12} {state}")

# --- Load every result row that exists ---
results = pd.concat([pd.read_parquet(p) for p in results_on_disk.values()],
                    ignore_index=True) if results_on_disk else pd.DataFrame()
print(f"\nResult rows loaded: {len(results):,}")
print(f"Models present: {sorted(results['model'].unique())}")

labels = pd.read_parquet(MASTER_DIR / "labels.parquet")
print(f"Labels: {labels.shape}")

### Helpers

`prevalence` comes from the folds themselves (`n_pos / n_test` summed over the
three test years), not from the whole base — the reference for AUPRC has to be
the prevalence of the set the model was scored on.

**Lift = AUPRC / prevalence.** For a rare disease the raw AUPRC is small by
construction; a model with AUPRC 0.17 where outbreaks are 0.28% of months is far
from a failure. Reporting AUPRC without its prevalence invites exactly that
misreading, so the two always travel together here.

In [ ]:
MODEL_LABELS = {
    "persistence": "Persistence", "climatology": "Climatology",
    "logreg": "Logistic regression", "ebm": "EBM",
    "rf": "Random Forest", "xgb": "XGBoost", "lgbm": "LightGBM",
}
MODEL_ORDER = ["persistence", "climatology", "logreg", "ebm", "rf", "xgb", "lgbm"]

def combo_rows(disease, definition, h=1):
    """Result rows for one combo, or an empty frame if it has not trained yet."""
    if results.empty:
        return pd.DataFrame()
    return results[(results["disease"] == disease)
                   & (results["definition"] == definition)
                   & (results["h"] == h)]

def prevalence_of(rows):
    """Positive rate across the test folds the models were actually scored on."""
    fold = rows.drop_duplicates(subset=["test_year"])[["n_pos", "n_test"]]
    return fold["n_pos"].sum() / fold["n_test"].sum() if len(fold) else np.nan

def performance_table(disease, definition, h=1):
    """AUROC, AUPRC and lift per model, averaged over the test years."""
    rows = combo_rows(disease, definition, h)
    if rows.empty:
        print(f"  MISSING: {disease}/{definition}/h{h} has not trained yet")
        return None, np.nan
    prev = prevalence_of(rows)
    table = (rows.groupby("model")[["auroc", "auprc"]].mean()
             .reindex([m for m in MODEL_ORDER if m in set(rows["model"])]))
    table["lift"] = table["auprc"] / prev
    table.index = [MODEL_LABELS.get(m, m) for m in table.index]
    table = table.round({"auroc": 3, "auprc": 3, "lift": 1})
    folds = sorted(rows["test_year"].unique())
    print(f"  {disease}/{definition}/h{h}: prevalence {prev:.3f}, "
          f"test years {folds}"
          + ("" if folds == TEST_YEARS else "   <- PARTIAL, not all folds present"))
    return table, prev

## Section 1 — Global performance by model (dengue, inc100)

Every model scored on the same folds, averaged over the three test years. This is
the paper's headline table.

Read AUPRC against the prevalence line, not against 1.0: a model guessing at
random scores exactly the prevalence, so the lift column is what says how much
signal there is.

In [ ]:
table3, prev_inc100 = performance_table("dengue", "inc100", h=1)
if table3 is not None:
    print(f"\nGlobal performance — dengue, inc100, h=1 (prevalence {prev_inc100:.3f})")
    print(table3.to_string())
    table3.to_csv(TABLES_DIR / "table3_global_dengue_inc100.csv")
    print(f"\nSaved: {TABLES_DIR.name}/table3_global_dengue_inc100.csv")
table3

## Section 2 — Global performance by model (dengue, endemic channel)

The same table under the endemic-channel definition, where an outbreak is a
departure from the municipality's own history for that month rather than a fixed
incidence level.

Watch the climatology row in particular. A seasonal baseline predicts what
normally happens in that month — which is precisely what the endemic channel
subtracts out — so it is blind to this kind of outbreak by construction, not by
poor fitting.

In [ ]:
table5, prev_canal = performance_table("dengue", "canal", h=1)
if table5 is not None:
    print(f"\nGlobal performance — dengue, endemic channel, h=1 "
          f"(prevalence {prev_canal:.3f})")
    print(table5.to_string())
    table5.to_csv(TABLES_DIR / "table5_global_dengue_canal.csv")
    print(f"\nSaved: {TABLES_DIR.name}/table5_global_dengue_canal.csv")
table5

## Section 3 — Onset versus continuation

**This is the paper's central claim, so the definitions matter more here than
anywhere else.** Aggregate metrics fuse two tasks of very different difficulty:
telling that an outbreak already under way will continue, and telling that a calm
municipality is about to enter one. Only the second anticipates anything.

The predictions carry the outbreak state at the **predicted** month (`y_true`).
What they do not carry is the state at the month the prediction was *made from*.
We recover it from the labels, joined on `(cod_ibge, ano, mes)` — the row's own
month — and call it `surto_atual`.

With both states in hand, every test row falls into one of four groups:

| Group | `surto_atual` | `y_true` | What it means |
|---|---|---|---|
| **Onset** | 0 | 1 | calm municipality entering an outbreak — *the task that matters* |
| **Continuation** | 1 | 1 | already in outbreak and staying there |
| Calm, stays calm | 0 | 0 | where a false alarm costs something |
| Outbreak ends | 1 | 0 | leaving an outbreak |

The metrics reported below, all at a **fixed decision threshold of 0.5**:

- **Onset recall** — of the calm→outbreak transitions, how many were alerted.
- **Continuation recall** — of the ongoing outbreaks, how many were alerted.
- **Precision restricted to calm municipalities** — of the alerts raised where the
  municipality was calm, how many turned into an outbreak. This is *not* the
  global precision: the global number is inflated by alerts on municipalities that
  were already in outbreak, which are easy and anticipate nothing.
- **False-alarm rate in calm months** — of calm months that stayed calm, how many
  got an alert.

The climatology baseline is measured on the **same slice** — same disease, same
definition, same horizon, same rows. Mixing definitions here once produced a
badly wrong baseline number in an earlier version of this work, so the slice is
passed explicitly and never aggregated away.

In [ ]:
DISEASE, DEFINITION, HORIZON = "dengue", "inc100", 1
THRESHOLD = 0.5

pred_path = PRED_DIR / f"{DISEASE}_{DEFINITION}_h{HORIZON}.parquet"
onset_table = None          # set below; stays None if this combo has not trained
if not pred_path.exists():
    print(f"MISSING: {pred_path.name} — this section needs it; skipping.")
else:
    preds = pd.read_parquet(pred_path)
    print(f"Predictions: {preds.shape}  models: {sorted(preds['model'].unique())}")

    # Outbreak state in the row's OWN month, from the labels.
    label_col = f"{DISEASE}_{DEFINITION}"
    current = (labels[["cod_ibge", "ano", "mes", label_col]]
               .rename(columns={label_col: "surto_atual"}))
    preds = preds.merge(current, on=["cod_ibge", "ano", "mes"], how="left",
                        validate="m:1")
    missing_state = preds["surto_atual"].isna().sum()
    print(f"Rows with no current-month label: {missing_state:,} "
          f"({missing_state / len(preds):.2%})")
    preds = preds.dropna(subset=["surto_atual"])
    preds["surto_atual"] = preds["surto_atual"].astype(int)

def phase_metrics(frame, threshold=THRESHOLD):
    """Split one model's predictions into the four phases and score each."""
    alert = frame["proba"] >= threshold
    calm = frame["surto_atual"] == 0
    ongoing = frame["surto_atual"] == 1
    onset = calm & (frame["y_true"] == 1)
    continuation = ongoing & (frame["y_true"] == 1)
    calm_stays = calm & (frame["y_true"] == 0)

    alerts_in_calm = alert & calm
    return {
        "n_onset": int(onset.sum()),
        "n_continuation": int(continuation.sum()),
        "recall_onset": alert[onset].mean() if onset.any() else np.nan,
        "recall_continuation": alert[continuation].mean() if continuation.any() else np.nan,
        "recall_global": alert[frame["y_true"] == 1].mean(),
        "precision_global": frame.loc[alert, "y_true"].mean() if alert.any() else np.nan,
        "precision_in_calm": frame.loc[alerts_in_calm, "y_true"].mean() if alerts_in_calm.any() else np.nan,
        "false_alarm_calm": alert[calm_stays].mean() if calm_stays.any() else np.nan,
    }

In [ ]:
if pred_path.exists():
    rows = []
    for model in ("rf", "lgbm", "xgb", "climatology", "persistence"):
        sub = preds[preds["model"] == model]
        if sub.empty:
            print(f"  {model}: not in this prediction file, skipped")
            continue
        rows.append({"model": MODEL_LABELS.get(model, model), **phase_metrics(sub)})

    onset_table = pd.DataFrame(rows).set_index("model")
    show = onset_table[["recall_onset", "recall_continuation", "recall_global",
                        "precision_global", "precision_in_calm", "false_alarm_calm"]]
    print(f"\nPhase-separated performance — {DISEASE}/{DEFINITION}/h{HORIZON}, "
          f"threshold {THRESHOLD}")
    print(f"Onset events: {onset_table['n_onset'].iloc[0]:,}   "
          f"continuation months: {onset_table['n_continuation'].iloc[0]:,}")
    print((show * 100).round(1).to_string())

    onset_table.to_csv(TABLES_DIR / "table_onset_vs_continuation.csv")
    print(f"\nSaved: {TABLES_DIR.name}/table_onset_vs_continuation.csv")

    # The contrast the paper is built on, stated in one place.
    if "Random Forest" in onset_table.index:
        rf_row = onset_table.loc["Random Forest"]
        print("\n" + "=" * 66)
        print("What the aggregate metric hides:")
        print(f"  Continuation recall : {rf_row['recall_continuation']:.1%}   (easy: already in outbreak)")
        print(f"  Onset recall        : {rf_row['recall_onset']:.1%}   (the task that anticipates)")
        print(f"  Gap                 : {abs(rf_row['recall_continuation']-rf_row['recall_onset'])*100:.1f} points")
        print(f"  Precision, all rows : {rf_row['precision_global']:.1%}")
        print(f"  Precision, calm only: {rf_row['precision_in_calm']:.1%}   (the honest number)")
        if "Climatology" in onset_table.index:
            clim = onset_table.loc["Climatology"]
            print(f"\n  Onset recall  RF {rf_row['recall_onset']:.1%}  vs  "
                  f"climatology {clim['recall_onset']:.1%}"
                  f"   (+{(rf_row['recall_onset']-clim['recall_onset'])*100:.1f} points)")
            print(f"  False alarm   RF {rf_row['false_alarm_calm']:.1%}  vs  "
                  f"climatology {clim['false_alarm_calm']:.1%}")
        print("=" * 66)
onset_table

### The same trade-off across every threshold

A single threshold is one point on a curve. Sweeping it shows the choice the
manager actually faces: alert more and catch more onsets at the cost of more false
alarms, or alert less and trust each alert more.

Two pairs are plotted. **All months** is the aggregate view — the flattering one.
**Calm municipalities only** restricts to alerts that anticipate something, and
its precision starts far lower. The distance between the two panels is the
paper's argument in a picture.

In [ ]:
import matplotlib.pyplot as plt

# Colours carry identity only; both pairs were checked for colour-vision
# separation (deutan dE 11.6, normal dE 25.7) rather than assumed to be safe.
C_PRECISION, C_COVERAGE = "#d95f02", "#1b9e77"
INK, MUTED, GRID = "#1a1a1a", "#5c5c5c", "#d9d9d9"

def sweep(frame, thresholds, restrict_to_calm):
    """Precision and coverage across thresholds, optionally only where calm."""
    out = []
    scope = frame[frame["surto_atual"] == 0] if restrict_to_calm else frame
    positives = scope["y_true"] == 1
    for t in thresholds:
        alert = scope["proba"] >= t
        out.append({
            "threshold": t,
            "precision": scope.loc[alert, "y_true"].mean() if alert.any() else np.nan,
            "coverage": alert[positives].mean() if positives.any() else np.nan,
        })
    return pd.DataFrame(out)

if pred_path.exists() and (preds["model"] == "rf").any():
    rf_preds = preds[preds["model"] == "rf"]
    grid = np.arange(0.05, 0.96, 0.05)
    panels = [("All months", sweep(rf_preds, grid, False)),
              ("Calm municipalities only", sweep(rf_preds, grid, True))]

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
    for ax, (title, curve) in zip(axes, panels):
        ax.plot(curve["threshold"], curve["precision"], color=C_PRECISION,
                linewidth=2, marker="o", markersize=4, label="Precision")
        ax.plot(curve["threshold"], curve["coverage"], color=C_COVERAGE,
                linewidth=2, marker="o", markersize=4, label="Coverage")
        ax.set_title(title, fontsize=10, color=INK)
        ax.set_xlabel("Decision threshold", fontsize=9, color=MUTED)
        ax.set_ylim(0, 1)
        ax.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.tick_params(colors=MUTED, labelsize=8)
    axes[0].set_ylabel("Proportion", fontsize=9, color=MUTED)
    axes[0].legend(frameon=False, fontsize=9, labelcolor=INK)
    fig.suptitle(f"Precision and coverage by threshold — {DISEASE}, {DEFINITION}, "
                 f"Random Forest", fontsize=11, color=INK, y=1.02)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"threshold_tradeoff.{ext}", bbox_inches="tight", dpi=200)
    print(f"Saved: figuras/threshold_tradeoff.pdf and .png")
    plt.show()

## Section 4 — Degradation by forecast horizon

One model per horizon, trained by direct forecasting: the features stay frozen at
month *t* and only the target moves to *t + h*. So a longer horizon does not give
the model more information to work with — it gives it the same information about a
month further away.

The fold split follows the **year of the target**, not of the observation, at every
horizon. That distinction is what keeps a row whose outbreak falls inside the test
year out of the training set.

In [ ]:
HORIZONS = [1, 2, 3, 4, 5, 6]
rows, missing = [], []
for h in HORIZONS:
    combo = combo_rows("dengue", "inc100", h)
    rf_rows = combo[combo["model"] == "rf"] if not combo.empty else combo
    if rf_rows.empty:
        missing.append(h)
        continue
    prev = prevalence_of(combo)
    rows.append({"h": h,
                 "auroc": rf_rows["auroc"].mean(),
                 "auprc": rf_rows["auprc"].mean(),
                 "lift": rf_rows["auprc"].mean() / prev,
                 "prevalence": prev,
                 "test_years": len(rf_rows)})

if missing:
    print(f"MISSING horizons (not trained yet): {missing}")

horizon_table = pd.DataFrame(rows).set_index("h") if rows else None
if horizon_table is not None:
    print("\nHorizon degradation — dengue, inc100, Random Forest")
    print(horizon_table.round(3).to_string())
    horizon_table.to_csv(TABLES_DIR / "table6_horizon_degradation.csv")
    print(f"\nSaved: {TABLES_DIR.name}/table6_horizon_degradation.csv")
horizon_table

In [ ]:
# --- Figure: the degradation curve ---
# Two series, one axis (both are areas under a curve, same 0-1 scale) — never a
# second y-axis. Blue/red checked for colour-vision separation (protan dE 21.1).
C_AUROC, C_AUPRC = "#1f77b4", "#d62728"

if horizon_table is not None and len(horizon_table) > 1:
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    ax.plot(horizon_table.index, horizon_table["auroc"], color=C_AUROC,
            linewidth=2, marker="o", markersize=6, label="AUROC")
    ax.plot(horizon_table.index, horizon_table["auprc"], color=C_AUPRC,
            linewidth=2, marker="s", markersize=6, label="AUPRC")

    # The prevalence line is the reference AUPRC would reach by chance; without it
    # the red curve looks like a collapse rather than a controlled decline.
    prev_line = horizon_table["prevalence"].mean()
    ax.axhline(prev_line, color=MUTED, linewidth=1, linestyle="--")
    ax.text(horizon_table.index.max(), prev_line + 0.012,
            f"chance level (prevalence {prev_line:.2f})",
            ha="right", fontsize=8, color=MUTED)

    ax.set_xlabel("Forecast horizon (months ahead)", fontsize=9, color=MUTED)
    ax.set_ylabel("Area under the curve", fontsize=9, color=MUTED)
    ax.set_ylim(0, 1)
    ax.set_xticks(list(horizon_table.index))
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK)
    ax.set_title("Degradation by horizon — dengue, inc100, Random Forest",
                 fontsize=11, color=INK)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"horizon_degradation.{ext}", bbox_inches="tight", dpi=200)
    print("Saved: figuras/horizon_degradation.pdf and .png")
    plt.show()
else:
    print("Not enough horizons trained yet to draw the curve.")

## Section 5 — Agreement between the four outbreak definitions

Cohen's kappa corrects the observed agreement for the agreement two labellers
would reach by chance. It matters here because most municipality-months are not
outbreaks, so raw agreement is high no matter what the definitions do — kappa
strips that trivial agreement out.

Computed on the labels themselves, over the test years, so it describes the
definitions and not any model.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from itertools import combinations

DEFINITIONS = ["inc100", "inc300", "canal", "zscore"]
disease = "dengue"

in_test = labels["ano"].isin(TEST_YEARS)
kappa = pd.DataFrame(index=DEFINITIONS, columns=DEFINITIONS, dtype=float)
raw_agreement = {}
for a, b in combinations(DEFINITIONS, 2):
    ya, yb = labels.loc[in_test, f"{disease}_{a}"], labels.loc[in_test, f"{disease}_{b}"]
    kappa.loc[a, b] = kappa.loc[b, a] = round(cohen_kappa_score(ya, yb), 3)
    raw_agreement[(a, b)] = (ya == yb).mean()
for d in DEFINITIONS:
    kappa.loc[d, d] = 1.0

print(f"Cohen's kappa between definitions — {disease}, test years {TEST_YEARS}")
print(kappa.to_string())
print("\nRaw agreement, for contrast (inflated by the shared non-outbreak months):")
for (a, b), value in raw_agreement.items():
    print(f"  {a:>7} vs {b:<7} {value:.1%} raw   vs kappa {kappa.loc[a, b]:+.3f}")

kappa.to_csv(TABLES_DIR / "table_kappa_definitions.csv")
print(f"\nSaved: {TABLES_DIR.name}/table_kappa_definitions.csv")
kappa

## Section 6 — Random Forest across diseases and definitions

Whether one definition serves every disease. Diseases still training are listed
rather than silently dropped, and a combination with no outbreak in the test years
is reported as such — an empty cell there is a property of the disease, not a
failure of the model.

In [ ]:
DISEASES = ["dengue", "zika", "chikungunya", "fa"]
PAIR_DEFS = ["inc100", "canal"]

rows, not_trained = [], []
for disease in DISEASES:
    for definition in PAIR_DEFS:
        combo = combo_rows(disease, definition, h=1)
        if combo.empty:
            not_trained.append(f"{disease}/{definition}")
            rows.append({"disease": disease, "definition": definition,
                         "prevalence": np.nan, "auroc": np.nan,
                         "auprc": np.nan, "lift": np.nan, "status": "not trained yet"})
            continue
        rf_rows = combo[combo["model"] == "rf"]
        prev = prevalence_of(combo)
        if rf_rows.empty or prev == 0 or np.isnan(prev):
            rows.append({"disease": disease, "definition": definition,
                         "prevalence": prev, "auroc": np.nan, "auprc": np.nan,
                         "lift": np.nan, "status": "no outbreak in test years"})
            continue
        rows.append({"disease": disease, "definition": definition,
                     "prevalence": prev,
                     "auroc": rf_rows["auroc"].mean(),
                     "auprc": rf_rows["auprc"].mean(),
                     "lift": rf_rows["auprc"].mean() / prev,
                     "status": "ok"})

disease_table = pd.DataFrame(rows).set_index(["disease", "definition"])
print("Random Forest by disease and definition (h=1, mean over test years)")
print(disease_table.round(3).to_string())
if not_trained:
    print(f"\nStill training: {', '.join(not_trained)}")

disease_table.to_csv(TABLES_DIR / "table4_disease_definition.csv")
print(f"\nSaved: {TABLES_DIR.name}/table4_disease_definition.csv")
disease_table

## Section 7 — What is complete and what is not

In [ ]:
print("Tables written:")
for path in sorted(TABLES_DIR.glob("*.csv")):
    print(f"  {path.name}")
print("\nFigures written:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(f"  {path.name}")

print(f"\nTraining coverage: {len(results_on_disk)} of 96 combos on disk")
for disease in DISEASES:
    n = len(by_disease.get(disease, ()))
    print(f"  {disease:<12} {n:>2}/24 combos"
          + ("  COMPLETE" if n == 24 else "  <- still running" if n else "  <- not started"))
print("\nEvery table above is built only from what was on disk when this ran.")